# Golden Dataset Evaluation
Compares trained model checkpoints on a held-out Golden Dataset - images sourced from google (24 of each produce type, 12 rotten; 12 healthy) independently from the training data to give an honest, out-of-distribution accuracy estimate.

Each model was trained under different dataset / augmentation  conditions; the golden set is fixed across all comparisons so the results are directly comparable.


| Model label | Dataset | Augmentation | Notes |
|---|---|---|---|
| Dirty | Full raw dataset (~29k images) | Yes | Includes near-duplicates and static pre-augmented images |
| No dups+aug | No duplicates | No | --- |
| No dups | No duplicates | No | Baseline clean dataset |

# EXPERIMENTS
### Success criteria
- ID val is over-saturated so selection is based on:
    
    `OOD validation accuracy (primary) => OOD AUC-ROC (Secondary) => OOD ECE (Tertiary).`

### Steps
1. Identify best dataset variation using EfficientNet STL baseline (deduplication, pre-augmentation)

    ***Deduplication was won by ECE tiebreake***
2. Identify best STL architecture at fixed optimizer (AdamW): EfficientNet vs Swin vs MaxViT
3. On winning arch, MTL vs STL with unified stopping criterion (primary-task loss)

4. On winning arch, architectural ablations:
    - freeze / partial-freeze / finetune
    - pretrained / random-init
    - class-weighted / unweighted loss
5. Augmentation ablation on best config from (4)
6. Post-hoc: temperature scaling on val > OOD ECE check

In [1]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from torchvision.models import get_model, get_weight
from torch.utils.data import DataLoader
from utils.dataset import ProduceDataset
from pathlib import Path
from safetensors.torch import load_file
sys.path.append("..")
from experiment_configs import task_2_config as experiments
from experiment_configs import task_2_config_final as experiments_f
from utils.mtl_model import MultiTaskClassifier
from sklearn.metrics import roc_curve, roc_auc_score, brier_score_loss, precision_recall_fscore_support,confusion_matrix
import numpy as np
from plotly.subplots import make_subplots

GOLDEN_PATH =  Path(".") / ".." / "golden_dataset3" 
NUM_HEALTH_CLASSES = 2
CLASS_NAMES = ["Healthy", "Rotten"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _load_state_dict(path):
    if path.endswith(".safetensors"):
        return load_file(path, device=str(device))
    ckpt = torch.load(path, weights_only=False, map_location=device)
    return ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt


def load_model(exp, path, num_produce_classes):
    """Rebuild architecture from exp config, then load weights."""
    pretrained_weights = get_weight(exp.weight_string)
    base = get_model(exp.architecture, weights=None)
    if exp.is_mtl:
    # MTL: wrap in MultiTaskClassifier (matches saved state_dict keys)
        model = MultiTaskClassifier(base, num_produce_classes=num_produce_classes,
                                    num_health_classes=NUM_HEALTH_CLASSES)
    else:
        # STL: replace final layer using head_attr pattern
        head_attr = "classifier" if hasattr(base, "classifier") else "head"
        head = getattr(base, head_attr)
        if isinstance(head, nn.Sequential):
            head[-1] = nn.Linear(head[-1].in_features, NUM_HEALTH_CLASSES)
        elif isinstance(head, nn.Linear):
            setattr(base, head_attr, nn.Linear(head.in_features, NUM_HEALTH_CLASSES))
        model = base

    model.load_state_dict(_load_state_dict(path))
    return model.to(device).eval(), pretrained_weights.transforms()


def evaluate(model, transforms, is_mtl):
    dataset = ProduceDataset(dataset_root_dir=GOLDEN_PATH, transform=transforms)
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    all_preds, all_labels, all_probs, all_p_rotten = [], [], [], []
    with torch.no_grad():
        for x, y_health, _ in loader:
            x, y_health = x.to(device), y_health.to(device)
            health_out = model(x)[0] if is_mtl else model(x)
            probs = torch.softmax(health_out, dim=1)
            preds = health_out.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y_health.cpu().tolist())
            all_probs.extend(probs.max(dim=1).values.cpu().tolist())
            all_p_rotten.extend(probs[:, 1].cpu().tolist())   # prob of positive (rotten) class

    return pd.DataFrame({
        "produce":    [Path(p).parent.name.split("__")[0] for p in dataset.image_paths],
        "true_label": all_labels,
        "pred":       all_preds,
        "correct":    [p == l for p, l in zip(all_preds, all_labels)],
        "confidence": all_probs,
        "p_rotten":   all_p_rotten,
    })

def compute_ece(labels, preds, confidences, n_bins=10):
    labels, preds, confidences = map(np.asarray, (labels, preds, confidences))
    edges = np.linspace(0, 1, n_bins + 1)
    ece, bins = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (confidences >= lo) & (confidences < hi) if i < n_bins - 1 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            bins.append({"conf": (lo + hi) / 2, "acc": None, "count": 0})
            continue
        bin_conf = confidences[mask].mean()
        bin_acc = (preds[mask] == labels[mask]).mean()
        bins.append({"conf": bin_conf, "acc": bin_acc, "count": int(mask.sum())})
        ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return ece, bins


In [2]:
# Define models to evaluate

MODELS = {
    # DETERMINE BEST DATASET VARIANT
    "Dirty":              (experiments.EX2_EFFICIENTNET_FINETUNE,         "1_select_dataset/EX2_dirty_efficientnet_finetune20260417190029.pth"),
    "Dedup_no_aug":       (experiments.EX2_EFFICIENTNET_FINETUNE,         "1_select_dataset/EX2_dedup_no_aug_EFFICIENTNET_FINETUNE_20260421113156.safetensors"),
    "Dedup_old_aug":      (experiments.EX2_EFFICIENTNET_FINETUNE,         "1_select_dataset/EX2_dedup_EFFICIENTNET_FINETUNE_20260421123836.safetensors"),
    "Dedup_new_aug_7":    (experiments.EX10_EFFICIENTNET_FINETUNE_AUG,    "1_select_dataset/EX10_EFFICIENTNET_FINETUNE_AUG_20260421211057.safetensors"),
    "Dedup_new_aug_10":   (experiments.EX11_EFFICIENTNET_FINETUNE_AUG,    "1_select_dataset/EX11_EFFICIENTNET_FINETUNE_AUG_20260421221546.safetensors"),
    "Dedup_new_aug_13":   (experiments.EX12_EFFICIENTNET_FINETUNE_AUG,    "1_select_dataset/EX12_EFFICIENTNET_FINETUNE_AUG_20260421210924.safetensors"),
    # DETERMINE BEST ARCHITECTURE
    "EffNet_s":             (experiments_f.EX1_EFFICIENTNET_FINETUNE,       "2_select_arch/EX1_EFFICIENTNET_FINETUNE_20260422022739.safetensors"),
    "EffNet_B4":            (experiments_f.EX1T_EFFICIENTNET_FINETUNE,      "2_select_arch/EX1T_EFFICIENTNET_FINETUNE_20260422032622.safetensors"),
    "Swin_s":               (experiments_f.EX2_SWIN_FINETUNE,               "2_select_arch/EX2_SWIN_FINETUNE_20260422100511.safetensors"),
    "Swin_s_43":               (experiments_f.EX2_SWIN_FINETUNE,               "2_select_arch/EX2_SWIN_FINETUNE_43_20260424013509.safetensors"),
    "Swin_s_44":               (experiments_f.EX2_SWIN_FINETUNE,               "best/EX2_SWIN_FINETUNE_44_20260424015207.safetensors"),
    "Swin_s_45":               (experiments_f.EX2_SWIN_FINETUNE,               "2_select_arch/EX2_SWIN_FINETUNE_45_20260424020906.safetensors"),
    "Swin_t":               (experiments_f.EX2T_SWIN_FINETUNE,              "2_select_arch/EX2T_SWIN_FINETUNE_20260422044721.safetensors"),
    "MaxViT":               (experiments_f.EX3_MAXVIT_FINETUNE,             "2_select_arch/EX3_MAXVIT_FINETUNE_20260422022737.safetensors"),
    # MTL Weighting experiment
    "ENET_90_mtl":          (experiments_f.EX4a_EFFICIENTNET_FINETUNE_MTL,  "3_mtl_comp/EX4a_EFFICIENTNET_FINETUNE_MTL_20260422113149.safetensors"),
    "ENET_75_mtl":          (experiments_f.EX4b_EFFICIENTNET_FINETUNE_MTL,  "3_mtl_comp/EX4b_EFFICIENTNET_FINETUNE_MTL_20260422145627.safetensors"),
    "ENET_50_mtl":          (experiments_f.EX4c_EFFICIENTNET_FINETUNE_MTL,  "3_mtl_comp/EX4c_EFFICIENTNET_FINETUNE_MTL_20260422141950.safetensors"),
    "MaxViT_90_mtl":        (experiments_f.EX5a_MAXVIT_FINETUNE_MTL,        "3_mtl_comp/EX5a_MAXVIT_FINETUNE_MTL_20260422163620.safetensors"),
    "MaxViT_75_mtl":        (experiments_f.EX5b_MAXVIT_FINETUNE_MTL,        "3_mtl_comp/EX5b_MAXVIT_FINETUNE_MTL_20260422112303.safetensors"),
    "MaxViT_50_mtl":        (experiments_f.EX5c_MAXVIT_FINETUNE_MTL,        "3_mtl_comp/EX5c_MAXVIT_FINETUNE_MTL_20260422125321.safetensors"),
    "swin_90_mtl":          (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "3_mtl_comp/EX6a_SWIN_FINETUNE_MTL_20260422201021.safetensors"), # Best
    "swin_75_mtl":          (experiments_f.EX6b_SWIN_FINETUNE_MTL,          "3_mtl_comp/EX6b_SWIN_FINETUNE_MTL_20260422203742.safetensors"),
    "swin_50_mtl":          (experiments_f.EX6c_SWIN_FINETUNE_MTL,          "3_mtl_comp/EX6c_SWIN_FINETUNE_MTL_20260422201057.safetensors"),
    # Best Ablations    
    "swin_freeze":          (experiments_f.EX7a_SWIN_MTL_FREEZE,            "4_ablations/EX7a_SWIN_MTL_FREEZE_20260422225028.safetensors"),                
    "swin_scratch":         (experiments_f.EX7b_SWIN_MTL_SCRATCH,           "4_ablations/EX7b_SWIN_MTL_SCRATCH_20260422231839.safetensors"),
    "swin_unweighted":      (experiments_f.EX7c_SWIN_MTL_UNWEIGHTED,        "4_ablations/EX7c_SWIN_MTL_UNWEIGHTED_20260422225054.safetensors"),
    # Variance tests
    "swin_90_mtl_seed43":       (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "5_variance_test/EX6a_SWIN_FINETUNE_MTL_43_20260423035309.safetensors"),
    "swin_90_mtl_seed44":       (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "5_variance_test/EX6a_SWIN_FINETUNE_MTL_44_20260423050145.safetensors"),
    "swin_90_mtl_seed45":       (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "5_variance_test/EX6a_SWIN_FINETUNE_MTL_45_20260423023125.safetensors"),
    # Augmentation + variance
    "swin_best_aug_seed1":       (experiments_f.EX8_SWIN_FINETUNE_MTL_AUG,          "6_augmentation/EX8_SWIN_FINETUNE_MTL_AUG_42_20260423121851.safetensors"),
    "swin_best_aug_seed2":       (experiments_f.EX8_SWIN_FINETUNE_MTL_AUG,          "6_augmentation/EX8_SWIN_FINETUNE_MTL_AUG_43_20260423133517.safetensors"),
    "swin_best_aug_seed3":       (experiments_f.EX8_SWIN_FINETUNE_MTL_AUG,          "6_augmentation/EX8_SWIN_FINETUNE_MTL_AUG_45_20260423121913.safetensors"),
#     # Low augmentation (spatial only)
    "swin_best_low_aug":                     (experiments_f.EX8a_SWIN_FINETUNE_MTL_AUG,         "6_augmentation/EX8a_SWIN_FINETUNE_MTL_AUG_20260423234629.safetensors" )
}

# Get produce count for MTL
TRAINING_DATA_DIR = Path(".") / "data" / "Fruit_And_Vegetable_Diseases_Dataset_no_identical_no_aug"
ds = ProduceDataset(dataset_root_dir=TRAINING_DATA_DIR)
num_produce_classes = ds.num_produce_types

# ── Run evaluation ─────────────────────────────────────────────────────────────
results = {}
for name, (exp, path) in MODELS.items():
    print(f"Evaluating: {name} ({exp.display_name})  (arch={exp.architecture}, mtl={exp.is_mtl})")
    model, transforms = load_model(exp, f"models/{path}", num_produce_classes)
    results[name] = evaluate(model, transforms, exp.is_mtl)
    print(f"  Overall: {results[name]['correct'].mean():.4f}")
import pickle
from pathlib import Path

# ── Config: name the current eval dataset so caches don't collide ────────────
EVAL_DATASET_NAME = "golden_3"        # change per run: "golden_2", "golden_3", ...
CACHE_DIR = Path("runs/eval") / EVAL_DATASET_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set True to force a full re-evaluation (ignores existing cache files)
FORCE_REEVAL = False

# ── Run evaluation with per-model caching ───────────────────────────────────
results = {}
for name, (exp, path) in MODELS.items():
    cache_path = CACHE_DIR / f"{name}.pkl"

    if cache_path.exists() and not FORCE_REEVAL:
        with open(cache_path, "rb") as f:
            results[name] = pickle.load(f)
        print(f"Loaded cached: {name}  ({results[name]['correct'].mean():.4f})")
        continue

    print(f"Evaluating: {name} ({exp.display_name})  "
          f"(arch={exp.architecture}, mtl={exp.is_mtl})")
    model, transforms = load_model(exp, f"models/{path}", num_produce_classes)
    df = evaluate(model, transforms, exp.is_mtl)
    results[name] = df

    with open(cache_path, "wb") as f:
        pickle.dump(df, f)
    print(f"  Overall: {df['correct'].mean():.4f}  (cached -> {cache_path})")

# Optional: also save a single combined file for convenience
with open(CACHE_DIR / "_all.pkl", "wb") as f:
    pickle.dump(results, f)
print(f"\nSaved combined results: {CACHE_DIR / '_all.pkl'}")


Evaluating: Dirty (EX2_EFFICIENTNET_FINETUNE)  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8297
Evaluating: Dedup_no_aug (EX2_EFFICIENTNET_FINETUNE)  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8581
Evaluating: Dedup_old_aug (EX2_EFFICIENTNET_FINETUNE)  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8554
Evaluating: Dedup_new_aug_7 (EX10_EFFICIENTNET_FINETUNE_AUG)  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8581
Evaluating: Dedup_new_aug_10 (EX11_EFFICIENTNET_FINETUNE_AUG)  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.7973
Evaluating: Dedup_new_aug_13 (EX12_EFFICIENTNET_FINETUNE_AUG)  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8405
Evaluating: EffNet_s (EX1_EFFICIENTNET_FINETUNE)  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8878
Evaluating: EffNet_B4 (EX1T_EFFICIENTNET_FINETUNE)  (arch=efficientnet_b4, mtl=False)
  Overall: 0.8770
Evaluating: Swin_s (EX2_SWIN_FINETUNE)  (arch=swin_s, mtl=False)
  Overall: 0.9000
Evaluating: Swin_s_43 (EX2_SWIN_FINETUNE)

In [3]:


# ── Compute all metrics first ─────────────────────────────────────────────────
metrics = {}
rows = []
for name, df in results.items():
    fpr, tpr, _ = roc_curve(df["true_label"], df["p_rotten"])
    auc = roc_auc_score(df["true_label"], df["p_rotten"])
    ece, bins = compute_ece(df["true_label"], df["pred"], df["confidence"])
    prec, rec, f1, _ = precision_recall_fscore_support(
        df["true_label"], df["pred"], average="binary", pos_label=1, zero_division=0
    )
    metrics[name] = {"fpr": fpr, "tpr": tpr, "auc": auc, "ece": ece, "bins": bins}
    rows.append({
        "Model":       name,
        "Accuracy":    df["correct"].mean(),
        "AUC-ROC":     auc,
        "ECE":         ece,
        "Brier":       brier_score_loss(df["true_label"], df["p_rotten"]),
        "Mean conf.":  df["confidence"].mean(),
        "Precision":   prec,
        "Recall":      rec,
        "F1":          f1,
        "N":           len(df),
    })

# ── Summary table (before figures) ────────────────────────────────────────────
summary = pd.DataFrame(rows).set_index("Model")
higher_better = ["Accuracy", "AUC-ROC", "F1", "Precision", "Recall"]
lower_better  = ["ECE", "Brier"]

def rank_highlight(col, ascending):
    """Color top 3 distinct values: dark > medium > light green."""
    ranks = col.rank(ascending=ascending, method='dense')
    colors = {
        1: 'background-color:#4C9B36; color:black; weight:bold;',  
        2: 'background-color:#97D586 ; color:black',             
        3: 'background-color:#B4E1A8; color:black',       
    }
    return [colors.get(int(r), '') for r in ranks]

display(summary.style
    .format("{:.4f}", subset=summary.columns.difference(["N"]))
    .apply(lambda c: rank_highlight(c, ascending=False), subset=higher_better)
    .apply(lambda c: rank_highlight(c, ascending=True),  subset=lower_better)
)

# ── Overall accuracy bar chart ─────────────────────────────────────────────────
overall = summary[["Accuracy"]].reset_index()
fig1 = px.bar(overall, x="Model", y="Accuracy", text_auto=".3f",
              title="Overall accuracy — golden dataset",
              color="Model", range_y=[0.5, 1.0])
fig1.update_traces(textposition="outside")
fig1.show()

# ── ROC curves ────────────────────────────────────────────────────────────────
fig_roc = go.Figure()
for name, m in metrics.items():
    fig_roc.add_trace(go.Scatter(x=m["fpr"], y=m["tpr"], mode="lines",
                                 name=f"{name} (AUC={m['auc']:.3f})"))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                             line=dict(dash="dash", color="gray"), name="Random"))
fig_roc.update_layout(title="ROC curves — golden dataset",
                      xaxis_title="False positive rate", yaxis_title="True positive rate",
                      height=500, width=700)
fig_roc.show()

# ── Reliability diagram ───────────────────────────────────────────────────────
# Reliability diagram — equal-mass (quantile) bins + marker size ∝ sample count
# Fixes the jaggedness from sparse equal-width bins; legend click still toggles each trace
N_BINS_VIS = 10
fig_ece = go.Figure()
for name, df in results.items():
    conf = df["confidence"].values
    correct = df["correct"].values.astype(int)
    edges = np.quantile(conf, np.linspace(0, 1, N_BINS_VIS + 1))
    edges[0], edges[-1] = 0.0, 1.0
    xs, ys, counts = [], [], []
    for i in range(N_BINS_VIS):
        lo, hi = edges[i], edges[i + 1]
        m = (conf >= lo) & (conf < hi) if i < N_BINS_VIS - 1 else (conf >= lo) & (conf <= hi)
        if m.sum() < 2:
            continue
        xs.append(float(conf[m].mean()))
        ys.append(float(correct[m].mean()))
        counts.append(int(m.sum()))
    fig_ece.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines+markers",
        marker=dict(size=[6 + float(np.sqrt(c)) * 1.5 for c in counts]),
        customdata=counts,
        hovertemplate="conf=%{x:.3f}<br>acc=%{y:.3f}<br>n=%{customdata}<extra></extra>",
        name=f"{name} (ECE={metrics[name]['ece']:.3f})",
    ))
fig_ece.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    line=dict(dash="dash", color="gray"), name="Perfect calibration"
))
fig_ece.update_layout(
    title="Reliability diagram (equal-mass bins, marker size ∝ sample count)",
    xaxis_title="Confidence", yaxis_title="Accuracy",
    xaxis=dict(range=[0.4, 1.0]), yaxis=dict(range=[0, 1]),
    height=550, width=900,
    legend=dict(itemclick="toggle", itemdoubleclick="toggleothers"),
)
fig_ece.show()


# ── Per-category grouped bar chart ────────────────────────────────────────────
# per_cat = []
# for name, df in results.items():
#     cat_acc = df.groupby("produce")["correct"].mean().reset_index()
#     cat_acc.columns = ["produce", "accuracy"]
#     cat_acc["model"] = name
#     per_cat.append(cat_acc)

# per_cat_df = pd.concat(per_cat)

# fig2 = px.bar(per_cat_df, x="produce", y="accuracy", color="model",
#               barmode="group", title="Per-category accuracy - golden dataset",
#               range_y=[0, 1.0], text_auto=".2f")
# fig2.update_layout(xaxis_tickangle=-45, height=500)
# fig2.show()

# ── Heatmap ───────────────────────────────────────────────────────────────────
# pivot = per_cat_df.pivot(index="model", columns="produce", values="accuracy")

# fig3 = px.imshow(pivot, text_auto=".2f", aspect="auto",
#                  color_continuous_scale="RdYlGn", range_color=[0.5, 1.0],
#                  title="Accuracy heatmap — model vs category")
# fig3.show()

# OOD golden-set accuracy - task-critical, this is what the system will actually face
# OOD AUC-ROC - robust to class imbalance, threshold-free. Great tiebreaker when accuracies are within 1-2pp
# OOD ECE/brier - calibration matters because ripeness grading downstream reads softmax confidence
# 
# 1. Identify best dataset variation using EfficientNet STL baseline (deduplication, pre-augmentation)
#    Deduplication (old aug) has wins over no aug; however, deduplicaiton + new aug has superior performance.
#    =DEDUPLICATION (NO AUG) WINS=
# 2. Identify best STL architecture at fixed optimizer (AdamW): EfficientNet vs Swin vs MaxViT
#    MAXVIT WINS BY ACC, ECE/BRIER (notably) & F1. Keep Enet_v2 for CNN comparison.
# 3. On winning arch, MTL vs STL with unified stopping criterion (primary-task loss)
# 4. On winning arch, architectural ablations:
#     - freeze / partial-freeze / finetune
#     - pretrained / random-init
#     - class-weighted / unweighted loss
# 5. Augmentation ablation on best config from (4)
# 6. Post-hoc: temperature scaling on val > OOD ECE check


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
Dirty,0.8297,0.9123,0.0848,0.1296,0.9112,0.8020,0.8757,0.8372,740
Dedup_no_aug,0.8581,0.9431,0.1116,0.1194,0.9676,0.8424,0.8811,0.8613,740
Dedup_old_aug,0.8554,0.9420,0.0629,0.1058,0.9183,0.8790,0.8243,0.8508,740
Dedup_new_aug_7,0.8581,0.9362,0.1007,0.1161,0.9527,0.8754,0.8351,0.8548,740
Dedup_new_aug_10,0.7973,0.9079,0.1253,0.1577,0.9226,0.8793,0.6892,0.7727,740
Dedup_new_aug_13,0.8405,0.9186,0.1024,0.1232,0.9429,0.8043,0.9000,0.8495,740
EffNet_s,0.8878,0.9606,0.0746,0.0937,0.9623,0.8889,0.8865,0.8877,740
EffNet_B4,0.8770,0.9437,0.0813,0.1033,0.9528,0.9067,0.8405,0.8724,740
Swin_s,0.9000,0.9724,0.0694,0.0804,0.9694,0.8700,0.9405,0.9039,740


In [4]:
def load_all_eval(root=Path("runs/eval")):
    results_by_dataset = {}
    for ds_dir in root.iterdir():
        if not ds_dir.is_dir():
            continue
        all_file = ds_dir / "_all.pkl"
        if all_file.exists():
            with open(all_file, "rb") as f:
                results_by_dataset[ds_dir.name] = pickle.load(f)
        else:
            # Fallback: load individual model files
            results_by_dataset[ds_dir.name] = {}
            for model_file in ds_dir.glob("*.pkl"):
                if model_file.stem.startswith("_"):
                    continue
                with open(model_file, "rb") as f:
                    results_by_dataset[ds_dir.name][model_file.stem] = pickle.load(f)
    return results_by_dataset

results_by_dataset = load_all_eval()
print(f"Loaded: {list(results_by_dataset.keys())}")
for ds_name, res in results_by_dataset.items():
    print(f"  {ds_name}: {len(res)} models")

# ── Helper: compute the summary for one dataset ──────────────────────────────
def build_summary(results_dict):
    """results_dict: {model_name: df_with_cols true_label/p_rotten/pred/confidence/correct}"""
    metrics, rows = {}, []
    for name, df in results_dict.items():
        fpr, tpr, _ = roc_curve(df["true_label"], df["p_rotten"])
        auc = roc_auc_score(df["true_label"], df["p_rotten"])
        ece, bins = compute_ece(df["true_label"], df["pred"], df["confidence"])
        prec, rec, f1, _ = precision_recall_fscore_support(
            df["true_label"], df["pred"], average="binary", pos_label=1, zero_division=0
        )
        metrics[name] = {"fpr": fpr, "tpr": tpr, "auc": auc, "ece": ece, "bins": bins}
        rows.append({
            "Model": name,
            "Accuracy": df["correct"].mean(),
            "AUC-ROC": auc,
            "ECE": ece,
            "Brier": brier_score_loss(df["true_label"], df["p_rotten"]),
            "Mean conf.": df["confidence"].mean(),
            "Precision": prec, "Recall": rec, "F1": f1,
            "N": len(df),
        })
    return pd.DataFrame(rows).set_index("Model"), metrics


# ── Load all three datasets' results ─────────────────────────────────────────
# results_by_dataset[ds_name] is a {model_name: df} dict, same shape as before

summaries  = {}   # {ds_name: summary_df}
metrics_by = {}   # {ds_name: metrics_dict}
for ds_name, res in results_by_dataset.items():
    summaries[ds_name], metrics_by[ds_name] = build_summary(res)


# ── Per-dataset tables (reuses your highlighting) ────────────────────────────
higher_better = ["Accuracy", "AUC-ROC", "F1", "Precision", "Recall"]
lower_better  = ["ECE", "Brier"]

def rank_highlight(col, ascending):
    ranks = col.rank(ascending=ascending, method='dense')
    colors = {
        1: 'background-color:#4C9B36; color:black; font-weight:bold;',
        2: 'background-color:#97D586; color:black',
        3: 'background-color:#B4E1A8; color:black',
    }
    return [colors.get(int(r), '') for r in ranks]

for ds_name, summary in summaries.items():
    print(f"\n=== {ds_name} ===")
    display(
        summary.style
            .format("{:.4f}", subset=summary.columns.difference(["N"]))
            .apply(lambda c: rank_highlight(c, ascending=False), subset=higher_better)
            .apply(lambda c: rank_highlight(c, ascending=True),  subset=lower_better)
    )


# ── Combined accuracy bar chart (grouped by model, colour by dataset) ────────
long = []
for ds_name, summary in summaries.items():
    for model_name, row in summary.iterrows():
        long.append({"Model": model_name, "Dataset": ds_name, "Accuracy": row["Accuracy"]})
long_df = pd.DataFrame(long)

fig_combined = px.bar(
    long_df, x="Model", y="Accuracy", color="Dataset",
    barmode="group", text_auto=".3f",
    title="Accuracy across golden datasets — all models",
    range_y=[0.4, 1.0],
    category_orders={
        # Optional: sort models by their golden_1 accuracy descending, keeps groups aligned
        "Model": summaries["golden_1"].sort_values("Accuracy", ascending=False).index.tolist(),
    },
)
fig_combined.update_traces(textposition="outside", textfont_size=9)
fig_combined.update_layout(height=600, width=max(900, 30 * len(long_df["Model"].unique())),
                           xaxis_tickangle=-60, legend_title="Dataset")
fig_combined.show()


Loaded: ['golden_1', 'golden_2', 'golden_3']
  golden_1: 33 models
  golden_2: 33 models
  golden_3: 33 models

=== golden_1 ===


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
Dirty,0.8408,0.9188,0.0845,0.1252,0.9102,0.8105,0.8800,0.8438,358
Dedup_no_aug,0.8715,0.9505,0.1107,0.1134,0.9765,0.8413,0.9086,0.8736,358
Dedup_old_aug,0.8799,0.9583,0.0425,0.0872,0.9212,0.8708,0.8857,0.8782,358
Dedup_new_aug_7,0.8715,0.9501,0.0858,0.1027,0.9502,0.8772,0.8571,0.8671,358
Dedup_new_aug_10,0.8128,0.9263,0.1086,0.1345,0.9215,0.8600,0.7371,0.7938,358
Dedup_new_aug_13,0.8464,0.9403,0.0966,0.1096,0.9430,0.8093,0.8971,0.8509,358
EffNet_s,0.9134,0.9802,0.0593,0.0704,0.9699,0.9138,0.9086,0.9112,358
EffNet_B4,0.9022,0.9617,0.0593,0.0808,0.9580,0.9070,0.8914,0.8991,358
Swin_s,0.9246,0.9840,0.0602,0.0668,0.9819,0.8814,0.9771,0.9268,358



=== golden_2 ===


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
Dirty,0.8258,0.9100,0.0969,0.1314,0.9146,0.8008,0.8694,0.8337,442
Dedup_no_aug,0.8484,0.9366,0.1137,0.1254,0.9621,0.8444,0.8559,0.8501,442
Dedup_old_aug,0.8439,0.9343,0.0748,0.1173,0.9165,0.8883,0.7883,0.8353,442
Dedup_new_aug_7,0.8484,0.9293,0.1121,0.1248,0.9545,0.8744,0.8153,0.8438,442
Dedup_new_aug_10,0.7941,0.8967,0.1440,0.1691,0.9247,0.8970,0.6667,0.7649,442
Dedup_new_aug_13,0.8371,0.9017,0.1065,0.1339,0.9436,0.8000,0.9009,0.8475,442
EffNet_s,0.8756,0.9480,0.0880,0.1061,0.9587,0.8778,0.8739,0.8758,442
EffNet_B4,0.8597,0.9330,0.0933,0.1175,0.9493,0.9040,0.8063,0.8524,442
Swin_s,0.8891,0.9662,0.0720,0.0857,0.9612,0.8681,0.9189,0.8928,442



=== golden_3 ===


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
Dirty,0.8297,0.9123,0.0848,0.1296,0.9112,0.8020,0.8757,0.8372,740
Dedup_no_aug,0.8581,0.9431,0.1116,0.1194,0.9676,0.8424,0.8811,0.8613,740
Dedup_old_aug,0.8554,0.9420,0.0629,0.1058,0.9183,0.8790,0.8243,0.8508,740
Dedup_new_aug_7,0.8581,0.9362,0.1007,0.1161,0.9527,0.8754,0.8351,0.8548,740
Dedup_new_aug_10,0.7973,0.9079,0.1253,0.1577,0.9226,0.8793,0.6892,0.7727,740
Dedup_new_aug_13,0.8405,0.9186,0.1024,0.1232,0.9429,0.8043,0.9000,0.8495,740
EffNet_s,0.8878,0.9606,0.0746,0.0937,0.9623,0.8889,0.8865,0.8877,740
EffNet_B4,0.8770,0.9437,0.0813,0.1033,0.9528,0.9067,0.8405,0.8724,740
Swin_s,0.9000,0.9724,0.0694,0.0804,0.9694,0.8700,0.9405,0.9039,740


## 2. Cross-Model Comparison
High-level comparison across all models: overall accuracy, per-category breakdown, and an accuracy heatmap. Key observation: the dirty dataset (largest volume) outperforms cleaner subsets, suggesting data volume dominates over data cleanliness for this task at current scale.

In [ ]:
for model_name, df in results.items():
    y_true = df["true_label"].values
    y_pred = df["pred"].values
    conf   = df["confidence"].values

    # ── Precompute all four panels ──
    cm = confusion_matrix(y_true, y_pred)

    categories = sorted(df["produce"].unique())
    rows = []
    for cat in categories:
        sub = df[df["produce"] == cat]
        p, r, f, _ = precision_recall_fscore_support(
            sub["true_label"], sub["pred"], average="binary", zero_division=0)
        rows.append({"Category": cat, "Precision": p, "Recall": r, "F1": f})
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    rows.append({"Category": "MACRO", "Precision": p_mac, "Recall": r_mac, "F1": f_mac})
    prf_df = pd.DataFrame(rows)

    errors = df[~df["correct"]].copy()
    errors["error_type"] = np.where(errors["true_label"] == 0, "H→R", "R→H")
    total = df.groupby("produce").size()
    breakdown = errors.groupby(["produce", "error_type"]).size().unstack(fill_value=0)
    breakdown_pct = (breakdown.div(total, axis=0) * 100).round(1)

    # ── Build 2x2 grid ──
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Confusion matrix", "Per-category P / R / F1",
                        "Confidence by outcome",  "Error direction per category (%)"),
        horizontal_spacing=0.12, vertical_spacing=0.15,
        column_widths=[0.35, 0.65],
    )

    # 1. Confusion matrix
    fig.add_trace(go.Heatmap(
        z=cm, x=CLASS_NAMES, y=CLASS_NAMES, colorscale="Blues",
        text=cm, texttemplate="%{text}", showscale=False,
    ), row=1, col=1)

    # 2. P/R/F1 per category (grouped bars)
    for metric, color in [("Precision", "#636EFA"), ("Recall", "#EF553B"), ("F1", "#00CC96")]:
        fig.add_trace(go.Bar(
            name=metric, x=prf_df["Category"], y=prf_df[metric],
            marker_color=color, legendgroup=metric,
        ), row=1, col=2)

    # 3. Confidence distribution — violin (plays nice with grouped barmode elsewhere)
    fig.add_trace(go.Violin(
        x=np.where(df["correct"], "Correct", "Incorrect"),
        y=conf, box_visible=True, meanline_visible=True,
        points="outliers", line_color="#333", fillcolor="#AAB7FF",
        showlegend=False,
    ), row=2, col=1)

    # 4. Error direction
    err_long = breakdown_pct.reset_index().melt(id_vars="produce", var_name="error_type", value_name="value")
    for etype, color in [("H→R", "#EF553B"), ("R→H", "#636EFA")]:
        sub = err_long[err_long["error_type"] == etype]
        fig.add_trace(go.Bar(
            name=etype, x=sub["produce"], y=sub["value"],
            marker_color=color, legendgroup=etype,
        ), row=2, col=2)

    fig.update_layout(
        height=750, width=1300,
        title_text=f"<b>{model_name}</b>",
        barmode="group",
    )
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    fig.update_xaxes(tickangle=-45, row=2, col=2)
    fig.update_yaxes(range=[0, 1.05], row=1, col=2)
    fig.update_yaxes(title_text="Confidence", row=2, col=1)

    fig.show()


In [ ]:
from scipy.optimize import minimize_scalar
import torch.nn.functional as F

# ── Temperature scaling calibration ───────────────────────────────────────────
# Calibrate the no-aug model (most overconfident).
# T is found by minimising NLL on the golden set — in practice you'd use a
# separate val set, but this demonstrates the technique on available data.

TARGET_MODEL = "No aug"

def evaluate_with_logits(model, transforms):
    """Same as evaluate() but also returns raw logits for calibration."""
    dataset = ProduceDataset(root_dir=GOLDEN_PATH, transform=transforms)
    loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            all_logits.append(model(imgs).cpu())
            all_labels.extend(labels.tolist())
    return torch.cat(all_logits), torch.tensor(all_labels)

# Reload the target model and collect logits
m, transforms = load_model(MODELS[TARGET_MODEL])
logits, labels = evaluate_with_logits(m, transforms)

# Find T that minimises NLL on the golden set
def nll(T):
    return F.cross_entropy(logits / T, labels).item()

result   = minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded")
best_T   = result.x
print(f"Optimal temperature T = {best_T:.3f}")

# ── Compare confidence distributions before and after ─────────────────────────
def conf_from_logits(logits, T=1.0):
    probs = torch.softmax(logits / T, dim=1)
    return probs.max(dim=1).values.numpy()

preds_orig = logits.argmax(dim=1).numpy()
correct    = (preds_orig == labels.numpy())

conf_before = conf_from_logits(logits, T=1.0)
conf_after  = conf_from_logits(logits, T=best_T)

fig_cal = make_subplots(rows=1, cols=2,
                        subplot_titles=["Before (T=1)", f"After (T={best_T:.2f})"])

for col, conf, title in [(1, conf_before, "Before"), (2, conf_after, "After")]:
    fig_cal.add_trace(go.Histogram(
        x=conf[correct],  name="Correct",   marker_color="#00CC96",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)
    fig_cal.add_trace(go.Histogram(
        x=conf[~correct], name="Incorrect", marker_color="#EF553B",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)

fig_cal.update_xaxes(range=[0.4, 1.0])
fig_cal.update_layout(barmode="overlay", title=f"{TARGET_MODEL} — confidence before vs after temperature scaling",
                      height=400)
fig_cal.show()

acc = correct.mean()
print(f"Accuracy before: {acc:.4f}")
print(f"Accuracy after:  {acc:.4f}  (unchanged — argmax is T-invariant)")

## 3. Per-Model Detailed Analysis
For each model: confusion matrix, per-category Precision / Recall / F1, prediction confidence distribution, and error direction breakdown (Healthy→Rotten vs Rotten→Healthy).

The confidence distribution reveals **calibration** — a well-calibrated model should show high confidence on correct predictions and lower confidence on errors. Error direction shows whether the model leans toward false positives (calling rotten produce healthy) or false negatives.